# Comparaison de modèles — HPP sévère

**Priorité** : recall. Tracking : [MLflow HF](https://thibautmodrin-mlflow.hf.space/).

1. Baseline sans rééquilibrage  
2. `class_weight` / SMOTE / UnderSampler  
3. LogReg / Random Forest / XGBoost  
4. Log MLflow + export `joblib` optionnel

Variables d'environnement utiles :
- `MLFLOW_TRACKING_URI` (défaut = Space HF)
- `HPP_EXPORT_ARTIFACTS=1` pour écrire `../app/model/artifacts/`
- `HPP_ENABLE_MLFLOW=0` pour désactiver le tracking

## 1. Setup

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

sys.path.insert(0, str(Path.cwd()))
from hpp_data import FEATURE_ORDER, TARGET, feature_lists_present, load_hpp

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.under_sampling import RandomUnderSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

ENABLE_MLFLOW = os.environ.get("HPP_ENABLE_MLFLOW", "1").strip().lower() not in ("0", "false", "no")
EXPORT_ARTIFACTS = os.environ.get("HPP_EXPORT_ARTIFACTS", "0").strip().lower() in ("1", "true", "yes")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "https://thibautmodrin-mlflow.hf.space")
EXPERIMENT = "HPP_Model_Comparison_Certification"

mlflow = None
if ENABLE_MLFLOW:
    try:
        import mlflow
        import mlflow.sklearn

        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment(EXPERIMENT)
        print("MLflow →", MLFLOW_URI, "| experiment:", EXPERIMENT)
    except Exception as e:
        print("MLflow indisponible:", e)
        mlflow = None
        ENABLE_MLFLOW = False


def build_preprocessor(quant, binary, nominal, ordinal):
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    nominal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    ordinal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric, quant),
        ("bin", "passthrough", binary),
        ("nom", nominal_pipe, nominal),
        ("ord", ordinal_pipe, ordinal),
    ])

## 2. Référence historique (si pas de re-run)

Chiffres issus des expériences archivées — à remplacer par le tableau §4 après une run sur `processed/`.

In [ ]:
results_ref = pd.DataFrame([
    {"modèle": "LogReg + SMOTE", "recall": 0.69, "precision": 0.08, "commentaire": "Réf. archive"},
    {"modèle": "Random Forest", "recall": 0.65, "precision": 0.09, "commentaire": "Réf. archive"},
    {"modèle": "XGBoost", "recall": 0.66, "precision": 0.09, "commentaire": "Réf. archive"},
])
results_ref

## 3. Données & split

In [ ]:
df, source = load_hpp()
df = df.dropna()
print(f"Source : {source} — après dropna : {df.shape}")

lists = feature_lists_present(df)
feat_cols = [c for c in FEATURE_ORDER if c in df.columns]
# garder l'ordre prod si possible
if set(feat_cols) != set(lists["quant"] + lists["binary"] + lists["nominal"] + lists["ordinal"]):
    feat_cols = lists["quant"] + lists["binary"] + lists["nominal"] + lists["ordinal"]

can_train = TARGET in df.columns and df[TARGET].nunique() >= 2 and len(df) >= 200
print("Entraînement possible :", can_train, "| n_features :", len(feat_cols))

if can_train:
    X = df[feat_cols]
    y = df[TARGET].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Train {X_train.shape} | Test {X_test.shape} | positifs train % {100*y_train.mean():.2f}")
else:
    print("Mode démo : lancez 00_Prepare_Data avec le .xls Drive pour entraîner + logger MLflow.")

## 4. Entraînement, métriques & MLflow

In [ ]:
def evaluate(pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    row = {
        "precision": round(float(precision_score(y_te, y_pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_te, y_pred, zero_division=0)), 4),
        "f1": round(float(f1_score(y_te, y_pred, zero_division=0)), 4),
    }
    if hasattr(pipe, "predict_proba") and y_te.nunique() > 1:
        proba = pipe.predict_proba(X_te)[:, 1]
        row["roc_auc"] = round(float(roc_auc_score(y_te, proba)), 4)
    return row


def log_run(name: str, pipe, metrics: dict, params: dict | None = None):
    if not ENABLE_MLFLOW or mlflow is None:
        return
    with mlflow.start_run(run_name=name):
        mlflow.log_param("data_source", source)
        mlflow.log_param("n_train", int(len(X_train)))
        mlflow.log_param("n_test", int(len(X_test)))
        if params:
            mlflow.log_params({k: str(v) for k, v in params.items()})
        mlflow.log_metrics(metrics)
        try:
            mlflow.sklearn.log_model(pipe, "model")
        except Exception as e:
            print("log_model skip:", e)


rows = []
fitted = {}

if can_train:
    q, b, n, o = lists["quant"], lists["binary"], lists["nominal"], lists["ordinal"]

    candidates = {
        "LogReg_baseline": Pipeline([
            ("preprocessor", build_preprocessor(q, b, n, o)),
            ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
        ]),
        "LogReg_balanced": Pipeline([
            ("preprocessor", build_preprocessor(q, b, n, o)),
            ("classifier", LogisticRegression(
                max_iter=2000, class_weight="balanced", random_state=42
            )),
        ]),
        "RandomForest": Pipeline([
            ("preprocessor", build_preprocessor(q, b, n, o)),
            ("classifier", RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            )),
        ]),
    }

    if HAS_IMBLEARN:
        candidates["LogReg_SMOTE"] = ImbPipeline([
            ("preprocessor", build_preprocessor(q, b, n, o)),
            ("sampler", SMOTE(random_state=42)),
            ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
        ])
        candidates["LogReg_Under"] = ImbPipeline([
            ("preprocessor", build_preprocessor(q, b, n, o)),
            ("sampler", RandomUnderSampler(random_state=42)),
            ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
        ])

    if HAS_XGB:
        spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        candidates["XGBoost"] = Pipeline([
            ("preprocessor", build_preprocessor(q, b, n, o)),
            ("classifier", XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                scale_pos_weight=float(spw),
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            )),
        ])

    for name, model in candidates.items():
        model.fit(X_train, y_train)
        metrics = evaluate(model, X_test, y_test)
        metrics["modèle"] = name
        rows.append(metrics)
        fitted[name] = model
        print(name, metrics)
        log_run(name, model, {k: v for k, v in metrics.items() if k != "modèle"})

    results_live = pd.DataFrame(rows).set_index("modèle").sort_values("recall", ascending=False)
    results_live
else:
    results_live = results_ref.set_index("modèle")
    print("Pas de run locale — tableau de référence :")
    results_live

## 5. Export artefact (optionnel)

Par défaut **désactivé** (`HPP_EXPORT_ARTIFACTS=0`) pour ne pas écraser le modèle Render par erreur.

```powershell
$env:HPP_EXPORT_ARTIFACTS = "1"
```

In [ ]:
ART = Path("..") / "app" / "model" / "artifacts"

if can_train and EXPORT_ARTIFACTS and fitted:
    # Préférer LogReg_SMOTE si présent, sinon meilleur recall
    prefer = "LogReg_SMOTE" if "LogReg_SMOTE" in fitted else results_live["recall"].idxmax()
    best = fitted[prefer]
    ART.mkdir(parents=True, exist_ok=True)
    joblib.dump(best, ART / "model.joblib")
    joblib.dump(feat_cols, ART / "feature_order.pkl")
    print(f"Exporté {prefer} → {ART}")
elif can_train and not EXPORT_ARTIFACTS:
    print("Artefacts non exportés (HPP_EXPORT_ARTIFACTS≠1). Modèle prod inchangé.")
else:
    print("Rien à exporter.")

## 6. Industrialisation

```
Drive → 00_Prepare → 01/02/03 ──► MLflow (tracking)
                              └─► joblib ──► FastAPI ──► Streamlit
```

| Couche | Lien |
|--------|------|
| API | https://hpp-api.onrender.com/docs |
| Streamlit | https://thibautmodrin-hpp-prediction.hf.space |
| MLflow | https://thibautmodrin-mlflow.hf.space/ |

**Après une run** : copier le tableau §4 dans le README / slides si les métriques changent.